In [ ]:
# Instalar geemap
!pip install geemap -q

import ee
import geemap
import time

# Autenticación e inicialización
try:
    ee.Initialize()
except Exception as e:
    print("Autenticando... Sigue las instrucciones en el enlace.")
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize(project='onyx-goal-465023-t8')

# --- 1. Área de interés ampliada (Córdoba, Colombia) ---
aoi = ee.Geometry.Rectangle([-76.0, 8.7, -75.8, 9.0])  # Área más grande

# --- 2. Fechas ajustadas (usando un año con datos garantizados) ---
start_date = '2020-01-01'
end_date = '2023-12-31'

# --- 3. Verificación de nombres de bandas ---
modis_ndvi = (
    ee.ImageCollection('MODIS/006/MOD13Q1')
    .filterDate(start_date, end_date)
    .filterBounds(aoi)
    .select('NDVI')  # Nombre correcto de la banda
)

# --- Chequear imágenes disponibles ---
image_list = modis_ndvi.toList(100)  # Convierte a lista para inspección
image_count = image_list.size().getInfo()
print(f"📊 Imágenes encontradas: {image_count}")

if image_count == 0:
    print("⚠️ No se encontraron imágenes. Probando alternativas...")

    # Alternativa 1: Usar Sentinel-3 (requiere escala diferente)
    print("Probando con Sentinel-3 SLSTR...")
    sentinel_ndvi = (
        ee.ImageCollection('COPERNICUS/S3/SLSTR')
        .filterDate('2023-01-01', '2023-01-31')  # Período corto para prueba
        .filterBounds(aoi)
        .select('NDVI')
    )
    if sentinel_ndvi.size().getInfo() > 0:
        print("✅ ¡Imágenes Sentinel-3 disponibles! Usando estos datos.")
        modis_ndvi = sentinel_ndvi
    else:
        # Alternativa 2: Mostrar datos de ejemplo
        print("Mostrando datos de ejemplo MODIS global...")
        modis_sample = ee.Image('MODIS/006/MOD13Q1/2023_01_01')
        ndvi_mean = modis_sample.select('NDVI')
else:
    # Procesamiento normal
    ndvi_mean = modis_ndvi.mean().clip(aoi)

# --- Visualización ---
Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(ndvi_mean, {'min': -1, 'max': 1, 'palette': ['red', 'yellow', 'green']}, 'NDVI')
Map.addLayer(aoi, {'color': 'blue'}, 'Área de estudio')
display(Map)

# --- Exportación (solo si hay datos) ---
if image_count > 0:
    task = ee.batch.Export.image.toDrive(
        image=ndvi_mean,
        description='NDVI_Export',
        folder='GEE_Exports',
        scale=250,
        region=aoi
    )
    task.start()
    print(f"🚀 Exportación iniciada (ID: {task.id})")
else:
    print("❌ No hay datos para exportar. Prueba ajustar parámetros.")

^C
